# 01 · Stage 1 — binary screening (healthy vs. unhealthy)

First gate of the three-stage cascade. Swin-V2-B at 384 px with a 2-way softmax
head, class-weighted cross-entropy, checkpoint selected on validation AUC.
The test split is never read here.

In [ ]:
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))   # or: pip install -e ..

import numpy as np, pandas as pd, torch
from fundus import config, labels
from fundus.data import (load_split, unhealthy_only, FundusDataset, make_loader,
                         train_transform, eval_transform, pos_weight)
from fundus.engine import set_seed, device, fit, predict

P = config.Paths()          # reads $FUNDUS_DATA_ROOT and $FUNDUS_RUN_ROOT
set_seed(config.SEED)
DEV = device()
print("device:", DEV, "| data:", P.data_root.resolve())

## Data

In [ ]:
tr, va = load_split(P.csv("train")), load_split(P.csv("val"))
assert not set(tr.image) & set(va.image), "train/val leakage"
print(pd.crosstab(tr.source, tr.Normal, margins=True))

cfg = config.STAGE1
train_ds = FundusDataset(tr, P.images("train"), train_transform(cfg.img_size), class_col="unhealthy")
val_ds   = FundusDataset(va, P.images("val"),   eval_transform(cfg.img_size),  class_col="unhealthy")
train_dl = make_loader(train_ds, cfg.batch_size, True,  cfg.num_workers)
val_dl   = make_loader(val_ds,   cfg.batch_size, False, cfg.num_workers)
print(f"train {len(train_ds)} | val {len(val_ds)}")

## Model and training

In [ ]:
from fundus.models import build_classifier
from fundus.engine import evaluate_binary

model = build_classifier(2, cfg)
counts = np.bincount(train_ds.labels, minlength=2)
class_w = torch.tensor(counts.sum() / (2.0 * np.maximum(counts, 1)), dtype=torch.float)
criterion = torch.nn.CrossEntropyLoss(weight=class_w)
print("class counts [healthy, unhealthy]:", counts.tolist(), "-> weights", class_w.numpy().round(3))

OUT = P.run("stage1")
hist = fit(model, train_dl, val_dl, criterion, evaluate_binary, OUT, cfg, monitors=("val_auc",))

## Operating point

A screening gate should favour sensitivity. The sweep below shows the
unhealthy/healthy recall trade-off; the chosen threshold is saved for the cascade.

In [ ]:
from fundus.models import load_weights
from sklearn.metrics import confusion_matrix

model = load_weights(model, OUT / "best_val_auc.pt", DEV)
m, (y, p, s) = evaluate_binary(model, val_dl, DEV)
print({k: round(v, 4) for k, v in m.items()})
print(confusion_matrix(y, (p >= 0.5).astype(int)))

for t in np.linspace(0.1, 0.9, 17):
    b = (p >= t).astype(int)
    print(f"thr {t:.2f}  unhealthy recall {b[y == 1].mean():.3f}  healthy recall {1 - b[y == 0].mean():.3f}")

S1_THRESHOLD = 0.5     # threshold used for the reported cascade
json.dump({"threshold": S1_THRESHOLD, "val_metrics": {k: float(v) for k, v in m.items()}},
          open(OUT / "operating_point.json", "w"), indent=2)